# Mão na Massa: Rede Neural — Desempenho no Trabalho

**Gabarito — Encontro 1, Bloco 2**

*Inteligência Artificial e Aprendizagem de Máquina · FECAP*

> Cenário: prever se uma pessoa teve bom desempenho numa tarefa, a partir de horas de sono e nível de estresse. Baseado na Lei de Yerkes-Dodson (estresse moderado maximiza desempenho -- nem baixo, nem alto demais).

## Passo 1 — Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

## Passo 2 — Carregar e explorar os dados

In [ ]:
df = pd.read_csv("desempenho_alunos.csv")
print(df.head())
print(df.isna().sum())

## Passo 3 — Tratar valores nulos

In [ ]:
df = df.dropna(subset=["nivel_estresse"])

## Passo 4 — Definir features (X) e alvo (y)

In [ ]:
X = df[["horas_sono", "nivel_estresse"]]
y = df["bom_desempenho"]

## Passo 5 — Separar treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5, stratify=y
)

## Passo 6 — Normalizar com StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

## Passo 7 — Treinar a Regressão Logística (baseline)

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(X_train_norm, y_train)

pred_log = log_reg.predict(X_test_norm)
print(f"Acuracia Regressao Logistica: {accuracy_score(y_test, pred_log):.2%}")

## Passo 8 — Treinar a Rede Neural

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(6,), activation="relu",
                     max_iter=8000, random_state=42)
mlp.fit(X_train_norm, y_train)

pred_mlp = mlp.predict(X_test_norm)
print(f"Acuracia Rede Neural: {accuracy_score(y_test, pred_mlp):.2%}")

## Passo 9 — Testar diferentes arquiteturas

In [ ]:
arquiteturas = [(2,), (4,), (6,), (8,), (12,), (8,4), (16,8)]
acuracias = []
for arq in arquiteturas:
    m = MLPClassifier(hidden_layer_sizes=arq, activation="relu",
                       max_iter=8000, random_state=42)
    m.fit(X_train_norm, y_train)
    acuracias.append(accuracy_score(y_test, m.predict(X_test_norm)))

plt.plot(range(len(arquiteturas)), acuracias, marker="o")
plt.xticks(range(len(arquiteturas)), [str(a) for a in arquiteturas], rotation=45)
plt.xlabel("arquitetura (neuronios por camada)")
plt.ylabel("acuracia no teste")
plt.tight_layout()
plt.show()

## Passo 10 — Avaliação final

In [ ]:
df_final = pd.read_csv("desempenho_final.csv").dropna(subset=["nivel_estresse"])
X_final = df_final[["horas_sono", "nivel_estresse"]]
y_final = df_final["bom_desempenho"]
X_final_norm = scaler.transform(X_final)

pred_final_mlp = mlp.predict(X_final_norm)
pred_final_log = log_reg.predict(X_final_norm)
print("Rede Neural:", accuracy_score(y_final, pred_final_mlp))
print("Reg. Logistica:", accuracy_score(y_final, pred_final_log))

### Gráfico sobre a diferença dos modelos Regressão Logística x Redes Neurais

In [ ]:
from sklearn.linear_model import LinearRegression

# Regressão Linear, usando só horas_sono, prevendo bom_desempenho como se fosse um número contínuo
lin_reg = LinearRegression().fit(df[["horas_sono"]], df["bom_desempenho"])

# A rede precisa das 2 entradas -- fixamos o estresse na média, pra poder plotar só variando o sono
estresse_medio = df["nivel_estresse"].mean()
sono_range = np.linspace(df["horas_sono"].min(), df["horas_sono"].max(), 200)

pontos_para_rede = pd.DataFrame({
    "horas_sono": sono_range,
    "nivel_estresse": estresse_medio
})
pontos_norm = scaler.transform(pontos_para_rede)
prob_rede = mlp.predict_proba(pontos_norm)[:, 1]

pred_linear = lin_reg.predict(sono_range.reshape(-1, 1))

# Plotando os 3 juntos
plt.figure(figsize=(9,5.5))
plt.scatter(df["horas_sono"], df["bom_desempenho"], alpha=0.3, color="gray", label="Dados reais")
plt.plot(sono_range, pred_linear, color="#B0413E", linewidth=2.5, label="Regressão Linear")
plt.plot(sono_range, prob_rede, color="#00A868", linewidth=2.5, label=f"Rede Neural (estresse fixo em {estresse_medio:.1f})")
plt.axhline(0, color="gray", linewidth=0.5, linestyle="--")
plt.axhline(1, color="gray", linewidth=0.5, linestyle="--")
plt.xlabel("horas de sono")
plt.ylabel("bom_desempenho")
plt.legend()
plt.tight_layout()
plt.show()

---
## Refletir

A Rede Neural bateu a Regressão Logística por uma margem grande aqui. Comparando com os outros 4 algoritmos (KNN, Árvore, Random Forest) já praticados no curso: esse ganho é parecido, maior ou menor? Por quê?

**Dica:** o ganho da rede tende a ser maior quando o padrão real tem uma curva/formato que uma reta não capta -- como o "círculo" da Lei de Yerkes-Dodson.